# 09 - k3s Networking Deep Dive

This notebook uses the Kubernetes Python client to inspect the live networking inside your local k3s cluster. It stays API-first: no manual `kubectl` parsing, no port-forwards, and no embedded secrets.

What it covers:
- active kubeconfig and API server context
- node, pod, Service, Endpoints, and EndpointSlice inventory
- service selector -> pod -> endpoint path mapping for the LLM stack
- k3s `LoadBalancer` internals via `svclb-*` pods in `kube-system`
- DNS names and in-cluster TCP/HTTP reachability from `python-toolbox`
- an optional watch loop for live EndpointSlice changes


In [ ]:
import sys
import importlib.metadata as metadata

required = ["kubernetes", "pandas", "matplotlib", "networkx"]
versions = {}
missing = []
for package in required:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = None
        missing.append(package)

print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
for package, version in versions.items():
    print(f"{package:12}: {version}")

if missing:
    print("Missing packages:", ", ".join(missing))
else:
    print("Dependency check passed.")


In [ ]:
from __future__ import annotations

import ipaddress
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from IPython.display import Markdown, display
from kubernetes import client, config, watch
from kubernetes.client import ApiException
from kubernetes.stream import stream

NAMESPACE = os.getenv("LLM_NAMESPACE", "llm-observability")
TARGET_SERVICES = ["langchain-demo", "ollama", "open-webui", "redis"]
CURRENT_WORKDIR = Path.cwd().resolve()
PROBE_TARGETS = [
    {"name": "langchain-demo", "host": f"langchain-demo.{NAMESPACE}.svc.cluster.local", "port": 8000, "mode": "http", "path": "/healthz"},
    {"name": "ollama", "host": f"ollama.{NAMESPACE}.svc.cluster.local", "port": 11434, "mode": "http", "path": "/api/tags"},
    {"name": "open-webui", "host": f"open-webui.{NAMESPACE}.svc.cluster.local", "port": 8080, "mode": "http", "path": "/health"},
    {"name": "redis", "host": f"redis.{NAMESPACE}.svc.cluster.local", "port": 6379, "mode": "tcp", "path": ""},
    {"name": "kubernetes-api", "host": "kubernetes.default.svc.cluster.local", "port": 443, "mode": "tcp", "path": ""},
]


def find_project_root(start: Path | None = None) -> Path | None:
    start = (start or CURRENT_WORKDIR).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Chart.yaml").exists() and (candidate / "values.local-k3s.yaml").exists():
            return candidate
        if candidate.name == "jupyter-notebooks" and (candidate.parent / "Chart.yaml").exists():
            return candidate.parent
    return None


def candidate_kubeconfigs() -> list[Path]:
    candidates: list[Path] = []
    env_value = os.getenv("KUBECONFIG")
    if env_value:
        for item in env_value.split(":"):
            path = Path(item).expanduser()
            if path.exists():
                candidates.append(path)
    for path in [Path("~/.kube/config").expanduser(), Path("/etc/rancher/k3s/k3s.yaml")]:
        if path.exists() and path not in candidates:
            candidates.append(path)
    return candidates


def load_k8s_config() -> tuple[str, str | None]:
    errors = []
    for kubeconfig_path in candidate_kubeconfigs():
        try:
            config.load_kube_config(config_file=str(kubeconfig_path))
            return "kubeconfig", str(kubeconfig_path)
        except Exception as exc:
            errors.append(f"{kubeconfig_path}: {exc}")
    try:
        config.load_incluster_config()
        return "in-cluster", None
    except Exception as exc:
        errors.append(f"in-cluster: {exc}")
    raise RuntimeError("Unable to load Kubernetes configuration. Tried: " + " | ".join(errors))


def ready_container_count(pod) -> int:
    return sum(1 for status in (pod.status.container_statuses or []) if status.ready)


def total_container_count(pod) -> int:
    return len(pod.spec.containers or [])


def selector_matches(labels: dict[str, str] | None, selector: dict[str, str] | None) -> bool:
    if not selector:
        return False
    labels = labels or {}
    return all(labels.get(key) == value for key, value in selector.items())


def port_string(port_obj) -> str:
    return f"{port_obj.port}->{port_obj.target_port}/{port_obj.protocol}"


def endpoint_addresses_for_service(service_name: str, endpoint_slices) -> list[str]:
    addresses: list[str] = []
    for endpoint_slice in endpoint_slices:
        labels = endpoint_slice.metadata.labels or {}
        if labels.get("kubernetes.io/service-name") != service_name:
            continue
        for endpoint in endpoint_slice.endpoints or []:
            addresses.extend(endpoint.addresses or [])
    return sorted(set(addresses))


def observed_prefixes(ip_values: list[str], prefixlen: int = 16) -> list[str]:
    prefixes = []
    for ip_value in ip_values:
        try:
            prefixes.append(str(ipaddress.ip_network(f"{ip_value}/{prefixlen}", strict=False)))
        except ValueError:
            continue
    return sorted(set(prefixes))


PROJECT_ROOT = find_project_root()
AUTH_MODE, KUBECONFIG_PATH = load_k8s_config()
core = client.CoreV1Api()
apps = client.AppsV1Api()
discovery = client.DiscoveryV1Api()
networking = client.NetworkingV1Api()
version_api = client.VersionApi()

active_context = None
if KUBECONFIG_PATH:
    try:
        _, active_context = config.list_kube_config_contexts(config_file=KUBECONFIG_PATH)
    except Exception:
        active_context = None

version_info = version_api.get_code()
print("Project root    :", PROJECT_ROOT)
print("Namespace       :", NAMESPACE)
print("Auth mode       :", AUTH_MODE)
print("Kubeconfig path :", KUBECONFIG_PATH)
print("Active context  :", (active_context or {}).get("name"))
print("Kubernetes ver. :", version_info.git_version)


## Cluster Snapshot

This cell inventories the live node and address space that your notebook will analyze. It is useful for confirming single-node k3s behavior, pod IP ranges, and service IP ranges before drilling into the app namespace.


In [ ]:
nodes = core.list_node().items
all_services = core.list_service_for_all_namespaces().items
all_pods = core.list_pod_for_all_namespaces().items

node_rows = []
for node in nodes:
    addresses = {address.type: address.address for address in (node.status.addresses or [])}
    node_rows.append(
        {
            "node": node.metadata.name,
            "os": node.status.node_info.os_image,
            "kernel": node.status.node_info.kernel_version,
            "container_runtime": node.status.node_info.container_runtime_version,
            "internal_ip": addresses.get("InternalIP"),
            "hostname": addresses.get("Hostname"),
        }
    )

service_ips = [svc.spec.cluster_ip for svc in all_services if getattr(svc.spec, "cluster_ip", None) and svc.spec.cluster_ip not in {"None", None}]
pod_ips = [pod.status.pod_ip for pod in all_pods if getattr(pod.status, "pod_ip", None)]

snapshot = {
    "node_count": len(nodes),
    "service_count": len(all_services),
    "pod_count": len(all_pods),
    "observed_service_prefixes": observed_prefixes(service_ips),
    "observed_pod_prefixes": observed_prefixes(pod_ips),
}

display(pd.DataFrame(node_rows))
snapshot


## Namespace Networking Inventory

The next cell uses the Kubernetes API directly to show the networking primitives in `llm-observability`: pods, Services, Endpoints, EndpointSlices, Ingresses, and NetworkPolicies. On your current stack, the interesting split is between internal `ClusterIP` services and the `open-webui` `LoadBalancer`.


In [ ]:
pods = core.list_namespaced_pod(NAMESPACE).items
services = core.list_namespaced_service(NAMESPACE).items
endpoints = core.list_namespaced_endpoints(NAMESPACE).items
endpoint_slices = discovery.list_namespaced_endpoint_slice(NAMESPACE).items
try:
    ingresses = networking.list_namespaced_ingress(NAMESPACE).items
except ApiException as exc:
    ingresses = []
    print(f"Ingress listing skipped: {exc.status} {exc.reason}")
try:
    netpols = networking.list_namespaced_network_policy(NAMESPACE).items
except ApiException as exc:
    netpols = []
    print(f"NetworkPolicy listing skipped: {exc.status} {exc.reason}")

pod_df = pd.DataFrame(
    [
        {
            "pod": pod.metadata.name,
            "phase": pod.status.phase,
            "ready": f"{ready_container_count(pod)}/{total_container_count(pod)}",
            "pod_ip": pod.status.pod_ip,
            "node": pod.spec.node_name,
            "labels": ", ".join(
                f"{k}={v}"
                for k, v in sorted((pod.metadata.labels or {}).items())
                if k in {"app.kubernetes.io/name", "app.kubernetes.io/instance"}
            ),
        }
        for pod in pods
    ]
).sort_values(["pod"]).reset_index(drop=True)

service_df = pd.DataFrame(
    [
        {
            "service": svc.metadata.name,
            "type": svc.spec.type,
            "cluster_ip": svc.spec.cluster_ip,
            "ports": ", ".join(port_string(port) for port in (svc.spec.ports or [])),
            "selector": svc.spec.selector or {},
            "external": [ing.ip or ing.hostname for ing in ((svc.status.load_balancer.ingress or []) if svc.status and svc.status.load_balancer else [])],
        }
        for svc in services
    ]
).sort_values(["service"]).reset_index(drop=True)

endpoint_df = pd.DataFrame(
    [
        {
            "service": endpoint.metadata.name,
            "addresses": [addr.ip for subset in (endpoint.subsets or []) for addr in (subset.addresses or [])],
            "ports": [f"{port.port}/{port.protocol}" for subset in (endpoint.subsets or []) for port in (subset.ports or [])],
        }
        for endpoint in endpoints
    ]
).sort_values(["service"]).reset_index(drop=True)

endpoint_slice_df = pd.DataFrame(
    [
        {
            "endpoint_slice": endpoint_slice.metadata.name,
            "service": (endpoint_slice.metadata.labels or {}).get("kubernetes.io/service-name"),
            "address_type": endpoint_slice.address_type,
            "addresses": [address for endpoint in (endpoint_slice.endpoints or []) for address in (endpoint.addresses or [])],
            "ports": [f"{port.name or port.port}:{port.port}/{port.protocol}" for port in (endpoint_slice.ports or [])],
        }
        for endpoint_slice in endpoint_slices
    ]
).sort_values(["service", "endpoint_slice"]).reset_index(drop=True)

display(Markdown(f"**Ingresses:** {len(ingresses)} | **NetworkPolicies:** {len(netpols)} | **Services:** {len(services)} | **Pods:** {len(pods)}"))
display(service_df)
display(pod_df)
display(endpoint_slice_df)
endpoint_df


## Service Path Matrix

This is the core service-routing view for the stack. For each Service, it shows the selector, which pods match it, the live endpoint IPs published to Endpoints/EndpointSlices, the Service DNS name, and any externally reachable address.


In [ ]:
service_path_rows = []
for svc in services:
    selector = svc.spec.selector or {}
    selected_pods = [pod for pod in pods if selector_matches(pod.metadata.labels, selector)]
    endpoint_ips = endpoint_addresses_for_service(svc.metadata.name, endpoint_slices)
    load_balancer_targets = [ing.ip or ing.hostname for ing in ((svc.status.load_balancer.ingress or []) if svc.status and svc.status.load_balancer else [])]
    node_ports = [port.node_port for port in (svc.spec.ports or []) if getattr(port, "node_port", None)]
    service_path_rows.append(
        {
            "service": svc.metadata.name,
            "dns": f"{svc.metadata.name}.{NAMESPACE}.svc.cluster.local",
            "type": svc.spec.type,
            "cluster_ip": svc.spec.cluster_ip,
            "selector": selector or None,
            "selected_pods": [pod.metadata.name for pod in selected_pods],
            "selected_pod_ips": [pod.status.pod_ip for pod in selected_pods],
            "ports": [port_string(port) for port in (svc.spec.ports or [])],
            "node_ports": node_ports,
            "endpoint_ips": endpoint_ips,
            "load_balancer_targets": load_balancer_targets,
        }
    )

service_paths_df = pd.DataFrame(service_path_rows).sort_values(["service"]).reset_index(drop=True)
display(service_paths_df)
focus_df = service_paths_df[service_paths_df["service"].isin(TARGET_SERVICES)].reset_index(drop=True)
focus_df


## Application Flow Interpretation

The Kubernetes API shows the network plumbing. This cell adds the higher-level application path used by this stack: `open-webui` receives north-south traffic, then talks east-west to `langchain-demo`, which proxies to `ollama`. `redis` provides local state inside the namespace.


In [ ]:
flow_rows = [
    {"from": "browser or host", "to": "open-webui service", "transport": "HTTP", "details": "LoadBalancer / ServiceLB entrypoint"},
    {"from": "open-webui service", "to": "open-webui pod", "transport": "ClusterIP -> pod IP", "details": "Service selector resolves to StatefulSet pod"},
    {"from": "open-webui pod", "to": "langchain-demo service", "transport": "HTTP", "details": "Ollama-compatible proxy path inside cluster"},
    {"from": "langchain-demo service", "to": "langchain-demo pod", "transport": "ClusterIP -> pod IP", "details": "Deployment-backed proxy"},
    {"from": "langchain-demo pod", "to": "ollama service", "transport": "HTTP", "details": "Inference request forwarded to local model runtime"},
    {"from": "ollama service", "to": "ollama pod", "transport": "ClusterIP -> pod IP", "details": "Deployment-backed model server"},
    {"from": "open-webui pod", "to": "redis service", "transport": "TCP", "details": "Internal cache / state path"},
]

pd.DataFrame(flow_rows)


## k3s ServiceLB Exposure

On k3s, `LoadBalancer` Services are implemented by the built-in ServiceLB controller, which creates `svclb-*` pods in `kube-system`. This cell maps those pods back to the owning Service so you can see the north-south entrypoint in API form.


In [ ]:
kube_system_pods = core.list_namespaced_pod("kube-system").items
load_balancer_services = [svc for svc in all_services if svc.spec.type == "LoadBalancer"]
svclb_rows = []

for svc in load_balancer_services:
    external_targets = [ing.ip or ing.hostname for ing in ((svc.status.load_balancer.ingress or []) if svc.status and svc.status.load_balancer else [])]
    related_svclb_pods = [
        pod
        for pod in kube_system_pods
        if (pod.metadata.labels or {}).get("svccontroller.k3s.cattle.io/svcname") == svc.metadata.name
        and (pod.metadata.labels or {}).get("svccontroller.k3s.cattle.io/svcnamespace") == svc.metadata.namespace
    ]
    for pod in related_svclb_pods or [None]:
        host_ports = []
        if pod is not None:
            for container_obj in pod.spec.containers or []:
                for port in container_obj.ports or []:
                    host_ports.append(f"{port.host_port}->{port.container_port}/{port.protocol}")
        svclb_rows.append(
            {
                "service": svc.metadata.name,
                "namespace": svc.metadata.namespace,
                "cluster_ip": svc.spec.cluster_ip,
                "external_targets": external_targets,
                "node_ports": [port.node_port for port in (svc.spec.ports or []) if getattr(port, "node_port", None)],
                "svclb_pod": pod.metadata.name if pod is not None else None,
                "svclb_pod_ip": pod.status.pod_ip if pod is not None else None,
                "svclb_node": pod.spec.node_name if pod is not None else None,
                "host_ports": host_ports,
            }
        )

svclb_df = pd.DataFrame(svclb_rows).sort_values(["namespace", "service"]).reset_index(drop=True)
svclb_df


## Network Graph

This visualization combines the Kubernetes routing primitives and the known application hops so the local network story is visible in one place. Solid edges represent API-backed relationships such as node hosting or Service selection; dashed edges represent application-level traffic between Services.


In [ ]:
graph = nx.DiGraph()
color_map = {
    "node": "#264653",
    "service": "#2a9d8f",
    "pod": "#e9c46a",
    "external": "#e76f51",
    "svclb": "#8ab17d",
}

for node in nodes:
    graph.add_node(f"node:{node.metadata.name}", kind="node", label=node.metadata.name)

for pod in pods:
    pod_node = f"pod:{pod.metadata.name}"
    graph.add_node(pod_node, kind="pod", label=pod.metadata.name)
    graph.add_edge(f"node:{pod.spec.node_name}", pod_node, relation="hosts")

for row in service_path_rows:
    svc_node = f"svc:{row['service']}"
    graph.add_node(svc_node, kind="service", label=row["service"])
    for pod_name in row["selected_pods"]:
        graph.add_edge(svc_node, f"pod:{pod_name}", relation="selects")
    for target in row["load_balancer_targets"]:
        ext_node = f"ext:{target}"
        graph.add_node(ext_node, kind="external", label=target)
        graph.add_edge(ext_node, svc_node, relation="north-south")

for _, row in svclb_df.iterrows():
    if pd.notna(row["svclb_pod"]):
        svclb_node = f"svclb:{row['svclb_pod']}"
        graph.add_node(svclb_node, kind="svclb", label=row["svclb_pod"])
        graph.add_edge(svclb_node, f"svc:{row['service']}", relation="service-lb")
        graph.add_edge(f"node:{row['svclb_node']}", svclb_node, relation="hosts")

logical_edges = [
    ("svc:open-webui", "svc:langchain-demo", "app hop"),
    ("svc:langchain-demo", "svc:ollama", "app hop"),
    ("svc:open-webui", "svc:redis", "app hop"),
]
for source, target, relation in logical_edges:
    if graph.has_node(source) and graph.has_node(target):
        graph.add_edge(source, target, relation=relation, style="dashed")

plt.figure(figsize=(16, 10))
positions = nx.spring_layout(graph, seed=42, k=1.2)
node_colors = [color_map[graph.nodes[node]["kind"]] for node in graph.nodes]
solid_edges = [edge for edge in graph.edges if graph.edges[edge].get("style", "solid") == "solid"]
dashed_edges = [edge for edge in graph.edges if graph.edges[edge].get("style") == "dashed"]

nx.draw_networkx_nodes(graph, positions, node_color=node_colors, node_size=2200, alpha=0.95)
nx.draw_networkx_labels(graph, positions, labels={node: graph.nodes[node]["label"] for node in graph.nodes}, font_size=8)
nx.draw_networkx_edges(graph, positions, edgelist=solid_edges, arrows=True, arrowstyle="-|>", width=1.6, alpha=0.7)
nx.draw_networkx_edges(graph, positions, edgelist=dashed_edges, arrows=True, arrowstyle="-|>", width=1.6, alpha=0.7, style="dashed")
nx.draw_networkx_edge_labels(graph, positions, edge_labels={edge: graph.edges[edge]["relation"] for edge in graph.edges}, font_size=7)
plt.title("Local k3s networking graph for llm-observability-stack")
plt.axis("off")
plt.show()


## In-Cluster DNS and Connectivity Probe

This cell execs into the running `python-toolbox` pod by using the Kubernetes streaming API. It resolves Service DNS names and performs TCP or HTTP checks from inside the cluster network namespace, which is the cleanest way to prove east-west connectivity without shelling out to `kubectl`.


In [ ]:
toolbox_pods = core.list_namespaced_pod(NAMESPACE, label_selector="app.kubernetes.io/name=python-toolbox").items
toolbox_pod = next((pod for pod in toolbox_pods if pod.status.phase == "Running"), None)

if toolbox_pod is None:
    raise RuntimeError("No running python-toolbox pod found. Keep pythonToolbox.enabled=true and re-run this cell.")

toolbox_container = toolbox_pod.spec.containers[0].name
probe_targets_json = json.dumps(PROBE_TARGETS, indent=2)
probe_script = f"""
import json
import socket
import urllib.request

targets = {probe_targets_json}
rows = []
for target in targets:
    record = {{
        "name": target["name"],
        "host": target["host"],
        "port": target["port"],
        "mode": target["mode"],
        "path": target["path"],
    }}
    try:
        infos = socket.getaddrinfo(target["host"], target["port"], type=socket.SOCK_STREAM)
        record["resolved_ips"] = sorted({{info[4][0] for info in infos}})
        record["dns_ok"] = True
    except Exception as exc:
        record["resolved_ips"] = []
        record["dns_ok"] = False
        record["error"] = str(exc)
        rows.append(record)
        continue
    try:
        if target["mode"] == "http":
            url = f"http://{{target['host']}}:{{target['port']}}{{target['path']}}"
            with urllib.request.urlopen(url, timeout=5) as response:
                record["connect_ok"] = True
                record["http_status"] = response.status
        else:
            with socket.create_connection((target["host"], target["port"]), timeout=5):
                record["connect_ok"] = True
    except Exception as exc:
        record["connect_ok"] = False
        record["error"] = str(exc)
    rows.append(record)
print(json.dumps(rows))
"""

raw = stream(
    core.connect_get_namespaced_pod_exec,
    toolbox_pod.metadata.name,
    NAMESPACE,
    command=["python", "-c", probe_script],
    container=toolbox_container,
    stderr=True,
    stdin=False,
    stdout=True,
    tty=False,
)

def parse_probe_output(text: str):
    import ast

    cleaned = (text or "").strip()
    if not cleaned:
        raise ValueError("python-toolbox probe returned empty output")

    candidates = [cleaned]
    list_start = cleaned.find("[")
    list_end = cleaned.rfind("]")
    if 0 <= list_start <= list_end:
        bracket_slice = cleaned[list_start : list_end + 1]
        if bracket_slice not in candidates:
            candidates.append(bracket_slice)

    errors = []
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError as exc:
            errors.append(f"json: {exc}")
        try:
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return parsed
        except Exception as exc:
            errors.append(f"literal_eval: {exc}")

    preview = cleaned[:500]
    joined_errors = " | ".join(errors[:4])
    raise ValueError(
        "Unable to parse python-toolbox probe output. "
        f"Raw preview: {preview!r}. Errors: {joined_errors}"
    )

try:
    probe_records = parse_probe_output(raw)
except ValueError:
    print("Raw exec output:")
    print(raw)
    raise

probe_df = pd.DataFrame(probe_records).sort_values(["name"]).reset_index(drop=True)
print("Toolbox pod      :", toolbox_pod.metadata.name)
print("Toolbox container:", toolbox_container)
probe_df


## Optional EndpointSlice Watch

Use this only when you want to observe live control-plane changes, for example while restarting a pod or scaling a Deployment in another terminal. It stays read-only unless you explicitly make changes elsewhere.


In [ ]:
RUN_ENDPOINT_WATCH = False
WATCH_SERVICE = "open-webui"
WATCH_SECONDS = 20

if not RUN_ENDPOINT_WATCH:
    print("Preview mode. Set RUN_ENDPOINT_WATCH=True to stream EndpointSlice updates.")
else:
    watcher = watch.Watch()
    seen = 0
    for event in watcher.stream(
        discovery.list_namespaced_endpoint_slice,
        namespace=NAMESPACE,
        label_selector=f"kubernetes.io/service-name={WATCH_SERVICE}",
        timeout_seconds=WATCH_SECONDS,
    ):
        obj = event["object"]
        addresses = [address for endpoint in (obj.endpoints or []) for address in (endpoint.addresses or [])]
        ports = [port.port for port in (obj.ports or [])]
        print(
            {
                "type": event["type"],
                "endpoint_slice": obj.metadata.name,
                "addresses": addresses,
                "ports": ports,
            }
        )
        seen += 1
        if seen >= 15:
            watcher.stop()


## Done

This notebook is the API-first networking companion to notebooks `01` through `08`. Use it when you want to explain or debug how traffic actually moves through your single-node k3s stack without depending on port-forward shells or manually parsing `kubectl` output.
